In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from shared.common import load_config, plot_setup, save_figure
from shared.helper_funcs import gaussian_cross_section, lattice_cross_section, rural_widths

In [ ]:
settings = load_config()
config = settings['kincaid']
inp, out = Path('../input'), Path('../output')
C0 = config['log_scale_ug_m3']
palette = plot_setup()
rng = np.random.default_rng(settings['seed'])

In [ ]:
raw = pd.read_csv(inp / 'kincaid.csv')

In [ ]:
# neutral/unstable hours + published range of the spread curves
data = raw.loc[raw['arc_km'].between(config['arc_min_km'], config['arc_max_km'])
               & raw['quality'].ge(config['quality_min']) & raw['stability'].between(1, 4) & raw['wind_m_s'].ge(1)
               & raw['observed_ug_m3'].ge(0) & raw['q_g_s'].gt(0)
               & raw['exit_m_s'].gt(0) & raw['stack_k'].gt(raw['ambient_k'])].copy()
x = 1000*data['arc_km'].to_numpy()
wind = data['wind_m_s'].to_numpy()
h = data['stack_height_m'].to_numpy()
# Briggs buoyancy-dominated neutral/unstable terminal rise
F = (9.80665*data['exit_m_s']*data['stack_diameter_m']**2
     * (data['stack_k']-data['ambient_k'])/(4*data['stack_k'])).to_numpy()
end_rise = np.where(F < 55, 49*F**(5/8), 119*F**(2/5))
final_rise = 1.6*F**(1/3)*end_rise**(2/3)/wind
past_rise = x >= end_rise
data = data.loc[past_rise].copy()
x, wind, h, F, end_rise, final_rise = [values[past_rise] for values in [x, wind, h, F, end_rise, final_rise]]
sy, sz = rural_widths(x, data['stability'].to_numpy())
receptor_z, center = np.zeros(len(data)), np.zeros(len(data))
flux = 1e6*data['q_g_s'].to_numpy()/wind  # g/s -> µg/s

In [ ]:
pred = data[['met_day', 'hour', 'arc_km', 'observed_ug_m3', 'q_g_s', 'wind_m_s', 'stability', 'quality', 'mixing_height_m']].copy()
pred['rise_m'] = final_rise
pred['terminal_distance_m'] = end_rise
pred['buoyancy_m4_s3'] = F
for variant, heights in [('stack', h), ('rise', h+final_rise)]:
    args = (center, receptor_z, heights, sy, sz)
    pred[f'gaussian_{variant}'] = flux*gaussian_cross_section(*args)
    pred[f'ca_{variant}'] = flux*lattice_cross_section(*args, config['dy_m'], config['dz_m'])
    pred[f'ca_{variant}_refined'] = flux*lattice_cross_section(*args, config['dy_m']/2, config['dz_m']/2)
assert np.isfinite(pred.drop(columns='mixing_height_m').select_dtypes('number')).all().all()
pred['effective_height_m'] = h+final_rise
assert (pred['ca_rise'] <= pred['ca_stack'] + 1e-10).all()

In [ ]:
models = {'gaussian_stack': 'Gaussian · stack height', 'gaussian_rise': 'Gaussian · Briggs rise',
          'ca_stack': 'CA · stack height', 'ca_rise': 'CA · Briggs rise'}
long = pred.melt(id_vars=['met_day', 'hour', 'arc_km', 'observed_ug_m3'], value_vars=list(models),
                 var_name='model', value_name='predicted_ug_m3')
long['sq_log1p'] = (np.log1p(long['predicted_ug_m3']/C0)-np.log1p(long['observed_ug_m3']/C0))**2
ratio = long['predicted_ug_m3']/long['observed_ug_m3'].replace(0, np.nan)
long['fac2'] = ratio.between(0.5, 2).astype(float).where(ratio.notna())
measures = ['sq_log1p', 'fac2', 'observed_ug_m3', 'predicted_ug_m3']
daily = long.groupby(['model', 'met_day', 'hour'])[measures].mean().groupby(level=['model', 'met_day']).mean()
scores = daily.groupby(level='model').mean().reset_index()
scores['rmse_log1p'] = np.sqrt(scores.pop('sq_log1p'))
scores['fractional_bias'] = 2*(scores['predicted_ug_m3']-scores['observed_ug_m3']) / (
    scores['predicted_ug_m3']+scores['observed_ug_m3'])
scores['days'] = pred['met_day'].nunique()
scores['arc_hours'] = len(pred)
scores['label'] = scores['model'].map(models)

In [ ]:
squared = daily['sq_log1p'].unstack('model')
draw = rng.integers(0, len(squared), size=(settings['field']['bootstrap_days'], len(squared)))
boot = pd.DataFrame(np.sqrt(squared.to_numpy()[draw].mean(axis=1)), columns=squared.columns)
center_rmse = np.sqrt(squared.mean())
contrasts = []
for candidate, ref in [('ca_rise', 'ca_stack'), ('gaussian_rise', 'gaussian_stack'), ('ca_rise', 'gaussian_rise')]:
    lo, hi = np.quantile(boot[candidate]-boot[ref], [0.025, 0.975])
    contrasts.append((candidate, ref, center_rmse[candidate]-center_rmse[ref], lo, hi))
paired = pd.DataFrame(contrasts, columns=['candidate', 'reference', 'delta_rmse_log1p', 'lo95', 'hi95'])
diagnostics = []
for variant in ['stack', 'rise']:
    ca = np.log1p(pred[f'ca_{variant}']/C0)
    analytic = np.log1p(pred[f'gaussian_{variant}']/C0)
    refined = np.log1p(pred[f'ca_{variant}_refined']/C0)
    diagnostics.append((variant, np.sqrt(np.mean((ca-analytic)**2)), np.sqrt(np.mean((ca-refined)**2))))
diagnostics = pd.DataFrame(diagnostics, columns=['variant', 'ca_gaussian_rms_log1p', 'refinement_rms_log1p'])
assert diagnostics['ca_gaussian_rms_log1p'].max() < 0.025

In [ ]:
pred.to_csv(out / 'kincaid_peaks.csv', index=False, float_format='%.7g')
scores.to_csv(out / 'metrics.csv', index=False, float_format='%.7g')
paired.to_csv(out / 'paired_days.csv', index=False, float_format='%.7g')
diagnostics.to_csv(out / 'lattice_gaussian_agreement.csv', index=False, float_format='%.7g')
display(scores[['label', 'days', 'arc_hours', 'rmse_log1p', 'fac2', 'fractional_bias']].round(3))
display(paired.round(4))
display(diagnostics.round(6))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.5), sharex=True, sharey=True, layout='constrained')
for ax, variant in zip(axes, ['stack', 'rise']):
    for i, (arc, points) in enumerate(pred.groupby('arc_km')):
        ax.scatter(np.log1p(points['observed_ug_m3']/C0), np.log1p(points[f'ca_{variant}']/C0),
                   label=f'{arc:g} km', marker=['o', 's', '^', 'D'][i],
                   facecolors='none', edgecolors=palette[i], linewidth=0.75, s=20, alpha=0.65)
    lim = np.log1p(pred[['observed_ug_m3', 'ca_stack', 'ca_rise']].to_numpy().max()/C0)*1.03
    ax.plot([0, lim], [0, lim], '--', color='0.4', lw=1)
    ax.set(title=models[f'ca_{variant}'], xlabel='Observed log(1 + C / C₀)',
           aspect='equal', xlim=(0, lim), ylim=(0, lim))
axes[0].set_ylabel('Modeled log(1 + C / C₀)')
axes[1].legend(fontsize=8)
sns.despine(fig=fig)
save_figure(fig, out, 'kincaid_sf6')
display(fig)
plt.close(fig)